# From one weight to handwritten digits: backpropagation from scratch

This notebook builds a neural network from nothing but Python arithmetic, in four steps:

1. **One weight.** What a gradient is, and why the chain rule is all backpropagation needs.
2. **A tiny network.** One input, two hidden nodes, one output. Why a step activation can't learn, and what fixes it.
3. **Matrices.** The same chain rule written for hundreds of weights at once, trained on images of handwritten digits.
4. **What a hidden layer buys you.** A linear model does surprisingly well, until it doesn't.

Nothing here uses a machine-learning library. Everything you need is plain Python, NumPy for Part 3 onwards, and `matplotlib` for pictures.

**Exercises** are marked in bold. Solution cells are tagged `solution`, so an instructor can strip them to produce the student version, e.g. with

    jupyter nbconvert --to notebook --TagRemovePreprocessor.remove_cell_tags=solution backprop.ipynb

In [ ]:
import math, random
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (6, 3.5)

## Part 1: One weight

### Background

A model is a function with knobs. The smallest possible one is

$$y = w\,x + b$$

with two knobs, $w$ and $b$. For a given input $x$ we would like the output $y$ to be close to a target $t$. We measure how far off we are with a **loss**:

$$L = (y - t)^2$$

Learning means turning the knobs to make $L$ smaller. The only question is: which way should each knob turn, and by how much?

### The gradient is "nudge and see"

Take $w$. If we nudge it by a tiny amount $h$ and recompute the loss, the loss changes by some amount. The ratio

$$\frac{L(w+h) - L(w)}{h}$$

tells us how much the loss changes *per unit change in $w$*. If it is positive, increasing $w$ makes things worse, so we should decrease $w$. If it is negative, increase $w$. That ratio is the **gradient** of the loss with respect to $w$. There is nothing more to it.

In [ ]:
x, t = 2.0, 7.0            # one input and its target
w, b = 0.5, 0.0            # starting values of the knobs

def loss(w, b):
    y = w*x + b
    return (y - t)**2

h = 1e-4
grad_w_numerical = (loss(w + h, b) - loss(w, b)) / h
grad_b_numerical = (loss(w, b + h) - loss(w, b)) / h
print("dL/dw ≈", round(grad_w_numerical, 3))
print("dL/db ≈", round(grad_b_numerical, 3))

### The chain rule gives the same number

Nudging works, but it needs two extra loss evaluations per knob. With a million knobs that is a million extra passes through the model. Calculus gives us the same number for free.

The loss depends on $w$ only *through* $y$. So

$$\frac{dL}{dw} = \frac{dL}{dy}\cdot\frac{dy}{dw}$$

Each factor is easy on its own:

- $\dfrac{dL}{dy} = 2(y - t)$: the derivative of a square.
- $\dfrac{dy}{dw} = x$: the derivative of $w x + b$ with respect to $w$.
- $\dfrac{dy}{db} = 1$.

So $\dfrac{dL}{dw} = 2(y-t)\,x$ and $\dfrac{dL}{db} = 2(y-t)$.

This is the whole idea of backpropagation: **start at the loss, walk backwards through the computation, multiply the local derivatives as you go.** Everything that follows is this, repeated.

In [ ]:
y = w*x + b
dL_dy = 2*(y - t)           # start at the loss
grad_w = dL_dy * x          # one step back to w
grad_b = dL_dy * 1          # one step back to b
print("chain rule:  dL/dw =", grad_w, "   dL/db =", grad_b)
print("numerical :  dL/dw ≈", round(grad_w_numerical, 3), "   dL/db ≈", round(grad_b_numerical, 3))

### Training is "take a small step downhill, repeat"

Once we know the gradient we move each knob a small step in the opposite direction. The step size is called the **learning rate**, usually written `lr`.

In [ ]:
w, b, lr = 0.5, 0.0, 0.05
history = []
for step in range(40):
    y = w*x + b                 # forward
    L = (y - t)**2
    dL_dy = 2*(y - t)           # backward
    w -= lr * dL_dy * x         # update
    b -= lr * dL_dy
    history.append(L)

plt.plot(history, marker=".")
plt.xlabel("step"); plt.ylabel("loss"); plt.title("Loss during training")
plt.show()
print(f"after training: w = {w:.3f}, b = {b:.3f}, y = {w*x + b:.3f}  (target {t})")

**Exercise 1.1.** Set `lr = 0.3` and run the training loop again. What happens, and why? (Think about what the update does when the gradient is large.)

**Exercise 1.2.** There are infinitely many pairs $(w, b)$ that give $y = 7$ when $x = 2$. Which one did training find? Does it depend on the starting values? Try starting from `w, b = 5.0, -1.0`.

## Part 2: A tiny network

### Background

A single weight can only draw a straight line. To draw anything more interesting we put a **non-linear function** between layers. The simplest is the step function:

$$\mathrm{step}(z) = \begin{cases}1 & z \ge 0\\ 0 & z < 0\end{cases}$$

Our network has one input $x$, two hidden nodes, and one linear output:

$$h_1 = \mathrm{step}(w_1 x + b_1),\qquad h_2 = \mathrm{step}(w_2 x + b_2),\qquad y = v_1 h_1 + v_2 h_2 + c$$

Each hidden node is a switch that flips on when $x$ passes a threshold ($x = -b_i / w_i$ when $w_i = 1$). The output adds up the switches that are on.

### The target: a staircase

We want the network to reproduce this curve:

$$t(x) = \begin{cases} 0 & x < 1 \\ 2 & 1 \le x < 3 \\ 1 & x \ge 3\end{cases}$$

Before running anything, convince yourself that the network *can* do it: switch 1 turns on at $x=1$ and contributes $+2$; switch 2 turns on at $x=3$ and contributes $-1$; the bias $c$ is $0$.

In [ ]:
def step(z):
    return 1.0 if z >= 0 else 0.0

def target(x):
    if x < 1: return 0.0
    if x < 3: return 2.0
    return 1.0

xs = [i / 10 for i in range(0, 50)]     # x from 0.0 to 4.9

def plot_fit(net, title):
    plt.step(xs, [target(x) for x in xs], where="post", label="target", lw=3, alpha=.4)
    plt.plot(xs, [net(x) for x in xs], label="network")
    plt.xlabel("x"); plt.ylabel("y"); plt.title(title); plt.legend(); plt.show()

### Part 2A: hand-set thresholds, learn the output weights

We fix the two switches at the right thresholds and let the network learn only $v_1, v_2, c$. Because the switches are just fixed on/off numbers, the gradient to the output weights is exactly the Part 1 formula: $\frac{dL}{dv_1} = 2(y-t)\,h_1$, and so on.

In [ ]:
w1, b1 = 1.0, -1.0          # switch 1 turns on at x = 1
w2, b2 = 1.0, -3.0          # switch 2 turns on at x = 3
v1, v2, c = 0.0, 0.0, 0.0   # to be learned
lr = 0.05

for epoch in range(100):
    for x in xs:
        h1 = step(w1*x + b1)           # forward
        h2 = step(w2*x + b2)
        y = v1*h1 + v2*h2 + c
        dL_dy = 2*(y - target(x))      # backward
        v1 -= lr * dL_dy * h1          # dy/dv1 = h1
        v2 -= lr * dL_dy * h2          # dy/dv2 = h2
        c  -= lr * dL_dy               # dy/dc  = 1

print(f"v1 = {v1:.2f}, v2 = {v2:.2f}, c = {c:.2f}   (expected 2, -1, 0)")
plot_fit(lambda x: v1*step(w1*x+b1) + v2*step(w2*x+b2) + c, "Part 2A: output weights learned")

### Part 2B: try to learn the thresholds too

Now suppose we don't know where the switches should be. We want to learn $w_1, b_1$ as well. The chain from the loss to $b_1$ has one more link than before, because the loss depends on $b_1$ through $z_1 = w_1 x + b_1$, then through $h_1 = \mathrm{step}(z_1)$, then through $y$:

$$\frac{dL}{db_1} = \frac{dL}{dy}\cdot\frac{dy}{dh_1}\cdot\frac{dh_1}{dz_1}\cdot\frac{dz_1}{db_1} = 2(y-t)\cdot v_1 \cdot \mathrm{step}'(z_1)\cdot 1$$

**Exercise 2.1.** What is $\mathrm{step}'(z)$, the slope of the step function, for $z \ne 0$? What does that mean for the product above?

Let's start switch 1 in the wrong place ($b_1 = -2$) and watch what happens.

In [ ]:
def dstep(z):
    return 0.0          # the step function is flat everywhere except at z = 0

w1, b1 = 1.0, -2.0      # wrong: switch 1 turns on at x = 2 instead of 1
w2, b2 = 1.0, -3.0
v1, v2, c = 0.0, 0.0, 0.0

for epoch in range(100):
    for x in xs:
        z1, z2 = w1*x + b1, w2*x + b2
        h1, h2 = step(z1), step(z2)
        y = v1*h1 + v2*h2 + c
        dL_dy = 2*(y - target(x))
        dz1 = dL_dy * v1 * dstep(z1)      # the extra link in the chain
        v1 -= lr*dL_dy*h1;  v2 -= lr*dL_dy*h2;  c -= lr*dL_dy
        w1 -= lr*dz1*x;     b1 -= lr*dz1

print(f"b1 = {b1:.2f}  (it started at -2.00 and never moved)")
plot_fit(lambda x: v1*step(w1*x+b1) + v2*step(w2*x+b2) + c, "Part 2B: threshold stuck")

### Part 2C: replace the step with something that has a slope

The fix is to use a smooth version of the step, the **sigmoid**:

$$\sigma_k(z) = \frac{1}{1 + e^{-kz}}$$

For large $k$ it looks almost exactly like a step, but its slope is never zero:

$$\sigma_k'(z) = k\,\sigma_k(z)\,(1 - \sigma_k(z))$$

Now every link in the chain is non-zero and the thresholds can move.

In [ ]:
k = 4.0
def sig(z):  return 1 / (1 + math.exp(-k*z))
def dsig(z): a = sig(z); return k * a * (1 - a)

zs = [i/20 for i in range(-60, 61)]
plt.plot(zs, [step(z) for z in zs], label="step")
plt.plot(zs, [sig(z) for z in zs], label=f"sigmoid, k={k}")
plt.legend(); plt.xlabel("z"); plt.title("Step vs sigmoid"); plt.show()

In [ ]:
random.seed(1)
w1, b1 = 1.0, -random.uniform(0, 5)     # random starting thresholds
w2, b2 = 1.0, -random.uniform(0, 5)
v1, v2, c = 0.0, 0.0, 0.0
print(f"starting thresholds: {-b1/w1:.2f}, {-b2/w2:.2f}")

for epoch in range(3000):
    for x in xs:
        z1, z2 = w1*x + b1, w2*x + b2      # forward
        h1, h2 = sig(z1), sig(z2)
        y = v1*h1 + v2*h2 + c
        dL_dy = 2*(y - target(x))          # backward
        dz1 = dL_dy * v1 * dsig(z1)
        dz2 = dL_dy * v2 * dsig(z2)
        v1 -= lr*dL_dy*h1;  v2 -= lr*dL_dy*h2;  c -= lr*dL_dy
        w1 -= lr*dz1*x;     b1 -= lr*dz1
        w2 -= lr*dz2*x;     b2 -= lr*dz2

print(f"learned thresholds:  {-b1/w1:.2f}, {-b2/w2:.2f}   (expected 1 and 3)")
plot_fit(lambda x: v1*sig(w1*x+b1) + v2*sig(w2*x+b2) + c, "Part 2C: everything learned")

**Exercise 2.2.** Design the weights by hand (no training) for a different target: $t(x) = 1$ for $1 \le x < 3$ and $0$ otherwise. Check your answer with `plot_fit`.

**Exercise 2.3.** In Part 2C, try `k = 1.0` and `k = 20.0`. Which trains better, and what does the fitted curve look like in each case? What does this suggest about the trade-off between "looks like a step" and "has a usable slope"?

**Exercise 2.4.** Add a third hidden node and find a target curve that needs it.

## Part 3: Matrices, and real data

### Background

The toy network has 7 numbers to learn. A network that reads an image has thousands, and a `for` loop over every weight in Python would take hours per pass. The fix is not a new idea but new notation: write the layer as a **matrix product**.

If the input is a row vector $\mathbf{x}$ with $n$ pixels, and the hidden layer has $m$ nodes, then all $m$ pre-activations at once are

$$\mathbf{z}_1 = \mathbf{x} W_1 + \mathbf{b}_1,\qquad \mathbf{h} = \sigma(\mathbf{z}_1),\qquad \mathbf{z}_2 = \mathbf{h} W_2 + \mathbf{b}_2$$

where $W_1$ is an $n\times m$ matrix and $W_2$ is $m \times 10$. NumPy does this with `x @ W1 + b1`, and it does it for a whole batch of images at once if `x` has one row per image.

### Ten outputs instead of one

We want to classify a digit as one of 10 classes, so the network has 10 output nodes. Instead of a squared error we use **softmax** to turn the 10 outputs into probabilities,

$$p_j = \frac{e^{z_j}}{\sum_i e^{z_i}}$$

and the **cross-entropy** loss $L = -\log p_{\text{correct}}$, which punishes putting low probability on the right answer. The reason to accept these two new formulas is that their combined gradient is remarkably simple:

$$\frac{\partial L}{\partial \mathbf{z}_2} = \mathbf{p} - \mathbf{t}$$

where $\mathbf{t}$ is the one-hot target vector (a 1 in the correct slot, 0 elsewhere). This is "output minus target", the same thing as the $2(y-t)$ we have used throughout, just without the factor 2.

### Backpropagation through a matrix layer

The chain rule works the same, but the local derivatives become matrix transposes. For a layer $\mathbf{z} = \mathbf{h} W + \mathbf{b}$ with gradient $\delta = \partial L/\partial\mathbf{z}$ arriving from above:

| quantity | scalar version (Part 2) | matrix version |
|---|---|---|
| gradient to the weights | `dz * h` | `h.T @ dz` |
| gradient to the bias | `dz` | `dz.sum(axis=0)` |
| gradient passed back to $\mathbf{h}$ | `dz * v` | `dz @ W.T` |
| through the sigmoid | `dh * dsig(z)` | `dh * h * (1 - h)` |

Each line is one of the toy-network lines, with every number replaced by an array.

### The data

`scikit-learn` ships with a small handwritten-digit dataset: 1797 images of 8×8 pixels, each labelled 0–9. It needs no download and trains in seconds, which makes it ideal for experimenting. (A section at the end shows how to switch to the full-size MNIST data.)

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
images = digits.images / 16.0        # pixel values 0..16 -> 0..1
labels = digits.target
print("images:", images.shape, "  labels:", labels.shape)

fig, axes = plt.subplots(2, 10, figsize=(10, 2.4))
for ax, im, lab in zip(axes.ravel(), images, labels):
    ax.imshow(im, cmap="gray_r"); ax.set_title(lab, fontsize=9); ax.axis("off")
plt.show()

# Hold out a quarter of the data that the network never sees during training.
img_train, img_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.25, random_state=0, stratify=labels)
X_train = img_train.reshape(len(img_train), -1)     # each 8x8 image -> row of 64 numbers
X_test  = img_test.reshape(len(img_test), -1)
print("X_train:", X_train.shape, "  X_test:", X_test.shape)

### The network as a few functions

We write one `train` function that handles both a network **with** a hidden layer and one **without** (`n_hidden=0`), so that we can compare them fairly later. Read the backward pass line by line against the table above.

In [ ]:
def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))   # subtract max: avoids overflow, same result
    return e / e.sum(axis=1, keepdims=True)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def train(X, y, n_hidden=0, epochs=200, lr=0.5, batch=32, seed=0, X_val=None, y_val=None):
    "Train a 1-hidden-layer network (or a linear model if n_hidden=0). Returns the parameters."
    rng = np.random.default_rng(seed)
    n_in, n_out = X.shape[1], 10
    T = np.eye(n_out)[y]                            # one-hot targets, one row per image
    if n_hidden:
        W1 = rng.normal(0, 1/np.sqrt(n_in), (n_in, n_hidden)); b1 = np.zeros(n_hidden)
        W2 = rng.normal(0, 1/np.sqrt(n_hidden), (n_hidden, n_out)); b2 = np.zeros(n_out)
    else:
        W1 = b1 = None
        W2 = rng.normal(0, 1/np.sqrt(n_in), (n_in, n_out)); b2 = np.zeros(n_out)

    history = []
    for epoch in range(epochs):
        order = rng.permutation(len(X))              # shuffle, then go through in small batches
        for i in range(0, len(X), batch):
            idx = order[i:i+batch]
            x, t = X[idx], T[idx]
            # ---- forward ----
            if n_hidden:
                h = sigmoid(x @ W1 + b1)
            else:
                h = x
            p = softmax(h @ W2 + b2)
            # ---- backward ----
            dz2 = (p - t) / len(idx)                 # "output minus target", averaged over the batch
            dW2 = h.T @ dz2                          # was  dz * h
            db2 = dz2.sum(axis=0)                    # was  dz
            if n_hidden:
                dh  = dz2 @ W2.T                     # was  dz * v
                dz1 = dh * h * (1 - h)               # through the sigmoid
                dW1 = x.T @ dz1
                db1 = dz1.sum(axis=0)
            # ---- update ----
            W2 -= lr * dW2;  b2 -= lr * db2
            if n_hidden:
                W1 -= lr * dW1;  b1 -= lr * db1
        if X_val is not None:
            history.append(accuracy((W1, b1, W2, b2), X_val, y_val))
    params = (W1, b1, W2, b2)
    return (params, history) if X_val is not None else params

def predict(params, X):
    W1, b1, W2, b2 = params
    h = sigmoid(X @ W1 + b1) if W1 is not None else X
    return softmax(h @ W2 + b2).argmax(axis=1)

def accuracy(params, X, y):
    return (predict(params, X) == y).mean()

### Train both models

First the **linear model**: 64 pixels straight to 10 outputs, no hidden layer. Then the same with **32 hidden nodes**.

In [ ]:
linear, hist_lin = train(X_train, y_train, n_hidden=0,  X_val=X_test, y_val=y_test)
hidden, hist_hid = train(X_train, y_train, n_hidden=32, X_val=X_test, y_val=y_test)

print(f"linear model  : train {accuracy(linear, X_train, y_train):.3f}   test {accuracy(linear, X_test, y_test):.3f}")
print(f"32 hidden     : train {accuracy(hidden, X_train, y_train):.3f}   test {accuracy(hidden, X_test, y_test):.3f}")

plt.plot(hist_lin, label="linear"); plt.plot(hist_hid, label="32 hidden")
plt.xlabel("epoch"); plt.ylabel("test accuracy"); plt.ylim(0.8, 1); plt.legend(); plt.show()

Both are around 97 %. That is a real network recognising real handwriting, written from scratch in a few dozen lines. But it also raises a question: **if the linear model is this good, what is the hidden layer for?**

### What the linear model has learned

A linear model's weights for class $j$ are a vector with one entry per pixel. Reshape it to 8×8 and it is an image: a **template**. Positive weights (dark) where that digit usually has ink, negative (light) where it usually doesn't.

In [ ]:
W = linear[2]
fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for j, ax in enumerate(axes):
    ax.imshow(W[:, j].reshape(8, 8), cmap="RdBu_r"); ax.set_title(j, fontsize=9); ax.axis("off")
plt.suptitle("Linear model: one weight template per digit", y=1.15); plt.show()

The linear model classifies by asking "which template does this image correlate with best?" On centred, size-normalised digits that works most of the time. What it cannot express is a rule like *"ink here means 9, unless there is also ink there, in which case 4"*. That needs an AND of features, which is a non-linear operation.

**Exercise 3.1.** Show the test digits that the linear model gets wrong, with the true and predicted labels. Are they the pairs you would expect to be confused?

**Exercise 3.2.** Train with `n_hidden` equal to 4, 8, 16, 64. How does test accuracy depend on the number of hidden nodes? Is more always better?

**Exercise 3.3.** The `train` function computes `dz2 = (p - t) / len(idx)`, the gradient *averaged* over the batch. Remove the division and retrain. After Exercise 1.1 you might expect the 32-times-larger steps to make training diverge. Does it? Why not? (Hint: how large can an entry of `p - t` be? How large could `2*(y - t)` get in Part 1?)

## Part 4: When template matching stops working

### Background

The linear model did well because every digit sits in the same place in the frame. Templates only work when the ink is where the template expects it. Let's break that assumption: paste each 8×8 digit at a **random position** inside a 12×12 frame. The digits are the same; only their location varies. A human would not notice a difference in difficulty.

In [ ]:
def shift_images(imgs, size=12, rng=None):
    "Paste each 8x8 image at a random offset inside a size x size frame."
    rng = rng or np.random.default_rng(0)
    out = np.zeros((len(imgs), size, size))
    for k, im in enumerate(imgs):
        r, c = rng.integers(0, size - 8 + 1, 2)
        out[k, r:r+8, c:c+8] = im
    return out

rng = np.random.default_rng(1)
# Training set: each image pasted at 4 different random positions (data augmentation).
img_train_s = np.vstack([shift_images(img_train, rng=rng) for _ in range(4)])
y_train_s   = np.tile(y_train, 4)
img_test_s  = shift_images(img_test, rng=rng)

fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for ax, im, lab in zip(axes, img_test_s, y_test):
    ax.imshow(im, cmap="gray_r"); ax.set_title(lab, fontsize=9); ax.axis("off")
plt.suptitle("Shifted digits", y=1.15); plt.show()

Xs_train = img_train_s.reshape(len(img_train_s), -1)
Xs_test  = img_test_s.reshape(len(img_test_s), -1)
print("training images:", Xs_train.shape)

In [ ]:
linear_s, hist_lin_s = train(Xs_train, y_train_s, n_hidden=0,  X_val=Xs_test, y_val=y_test)
hidden_s, hist_hid_s = train(Xs_train, y_train_s, n_hidden=64, X_val=Xs_test, y_val=y_test)

print(f"linear model  : test {accuracy(linear_s, Xs_test, y_test):.3f}")
print(f"64 hidden     : test {accuracy(hidden_s, Xs_test, y_test):.3f}")

plt.plot(hist_lin_s, label="linear"); plt.plot(hist_hid_s, label="64 hidden")
plt.xlabel("epoch"); plt.ylabel("test accuracy"); plt.ylim(0, 1); plt.legend()
plt.title("Shifted digits"); plt.show()

Same digits, same training code, same amount of data. The linear model collapses to around 30 % because no single template can match a "3" in sixteen different places at once; the templates blur into mush. The network with a hidden layer loses a few points and carries on.

Why? Look at what the hidden nodes have become.

In [ ]:
W1 = hidden_s[0]
fig, axes = plt.subplots(4, 16, figsize=(12, 3.2))
for j, ax in enumerate(axes.ravel()):
    ax.imshow(W1[:, j].reshape(12, 12), cmap="RdBu_r"); ax.axis("off")
plt.suptitle("The 64 hidden nodes as 12×12 weight images", y=1.02); plt.show()

W_lin = linear_s[2]
fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for j, ax in enumerate(axes):
    ax.imshow(W_lin[:, j].reshape(12, 12), cmap="RdBu_r"); ax.set_title(j, fontsize=9); ax.axis("off")
plt.suptitle("For comparison: the linear model's templates, now blurred", y=1.15); plt.show()

The hidden nodes are no longer whole-digit templates. Many are **local stroke detectors**: a short bar, a curve, an edge, at a particular place. The output layer then combines them: "a vertical bar somewhere on the left AND a loop above it" can be a 9 regardless of exactly where the digit sits, because there are several detectors for each stroke at several positions. Nobody told the network to do this. It fell out of the chain rule.

That is the power of a hidden layer in one sentence: **it lets the model build its own features and combine them non-linearly, instead of correlating with a fixed template.**

### Exercises

**Exercise 4.1.** Vary the frame size (`size=9`, `10`, `12`, `14`) and record the test accuracy of both models. Plot accuracy against frame size. Where does the linear model start to fail?

**Exercise 4.2.** With the 12×12 frame, train the hidden-layer model with 1, 4 and 8 random copies of each training image. How much does extra data help the hidden model? Does it help the linear model at all?

**Exercise 4.3.** The classic case where a linear model is hopeless is XOR: four points $(0,0)\to 0$, $(0,1)\to 1$, $(1,0)\to 1$, $(1,1)\to 0$. Use `train` with `n_hidden=0` and then `n_hidden=2` (treat the labels as classes 0 and 1; the other 8 output nodes are simply never used). Explain geometrically why one is impossible.

**Exercise 4.4 (harder).** Our hidden layer uses a sigmoid. Modern networks use ReLU, $\max(0, z)$, whose derivative is 1 for $z>0$ and 0 otherwise. Change `train` to use ReLU (both the forward and the backward line) and compare. You may need a smaller learning rate.

## Going further: the full MNIST dataset

Everything above transfers unchanged to the classic 28×28 MNIST digits (70 000 images). The only differences are the shape (784 inputs) and the time per epoch (a few seconds in NumPy). Download it once with scikit-learn and reuse the exact same `train` function, with a larger hidden layer and fewer epochs:

```python
from sklearn.datasets import fetch_openml
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X = mnist.data / 255.0
y = mnist.target.astype(int)
X_train, X_test, y_train, y_test = X[:60000], X[60000:], y[:60000], y[60000:]

linear = train(X_train, y_train, n_hidden=0,   epochs=10, lr=0.5)
hidden = train(X_train, y_train, n_hidden=128, epochs=10, lr=0.5)
print(accuracy(linear, X_test, y_test), accuracy(hidden, X_test, y_test))
```

Expect roughly 92 % for the linear model and 97–98 % with 128 hidden nodes. On full-size digits the gap is visible even without shifting, because 28×28 handwriting varies far more in shape than the tiny 8×8 digits do. The same experiments as in Parts 3 and 4 (templates, misclassified digits, shifted frames) are worth repeating here; the weight images in particular are much more recognisable at 28×28.

## Summary

- A **gradient** is "nudge a knob, see how the loss changes". The **chain rule** computes it without nudging, by multiplying local derivatives backwards from the loss.
- **Backpropagation** is nothing more than the chain rule applied to every weight, organised so that each layer's gradient reuses the layer above.
- A **step activation** can't be trained because it has no slope; **sigmoid** and **ReLU** fix that.
- A **linear model** is a template matcher. On well-aligned data it can be very good, which is why ~90 % is often easy.
- A **hidden layer** lets the network invent its own features (strokes, edges) and combine them non-linearly, which is what the remaining accuracy needs, and what makes it robust to variation the template can't absorb.